# SyLOC-T : Analyse de Donnees Avancees (BDA) & Deploiement Cloud
### CROUS de Thies - Gestion du Patrimoine Domanial

Ce notebook presente le pipeline complet d'analyse de donnees pour le projet SyLOC-T :
- Initialisation et gestion de la base de donnees
- Requetes SQL avancees et visualisations statistiques (KPIs)
- Cartographie geospatiale interactive des locaux du campus
- Modele de Machine Learning : prediction des retards de paiement
- Export automatique des rapports comptables et patrimoniaux
- Exposition securisee de l'API Django avec liens directs formates

### 1. Clonage du depot et installation des dependances

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub Token (laisser vide si public) : ").strip()

!rm -rf SyLOC-T
if token:
    !git clone -b develop https://{token}@github.com/mhdlamine21/SyLOC-T.git
else:
    !git clone -b develop https://github.com/mhdlamine21/SyLOC-T.git

%cd SyLOC-T

!pip install -r vcn_backend/requirements.txt
!pip install pandas matplotlib seaborn folium scikit-learn openpyxl

### 2. Initialisation de la base de donnees (SQLite)

In [ ]:
import os

os.environ["DB_ENGINE"] = "sqlite"
with open("vcn_backend/.env", "w") as f:
    f.write("SECRET_KEY=django-insecure-colab-key-syloc\n")
    f.write("DEBUG=True\n")
    f.write("DB_ENGINE=sqlite\n")
    f.write("ALLOWED_HOSTS=*\n")
    f.write("CORS_ALLOWED_ORIGINS=http://localhost:5173,http://localhost:3000\n")
    f.write("CSRF_TRUSTED_ORIGINS=http://localhost:3000,http://localhost:5173,http://127.0.0.1:5173,http://localhost:8000,http://127.0.0.1:8000,https://*.trycloudflare.com,https://*.loca.lt,https://*.ngrok-free.app,https://*.ngrok.io\n")

%cd vcn_backend
!python manage.py migrate
!python seed.py
%cd ..

### 3. Requetes SQL BDA & Indicateurs de Performance (KPIs)

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect('vcn_backend/db.sqlite3')

# Requete 1 : Etat et repartition du parc immobilier
query_locaux = """
SELECT 
    type_local AS Type,
    CASE WHEN est_libre = 1 THEN 'Disponible' ELSE 'Occupe' END AS Disponibilite,
    etat_physique AS Etat,
    gestionnaire AS Gestionnaire,
    COUNT(*) AS Total,
    ROUND(AVG(surface_m2), 1) AS Surface_Moy_m2
FROM patrimoine_local
GROUP BY type_local, est_libre, etat_physique, gestionnaire
ORDER BY Total DESC;
"""
df_locaux = pd.read_sql_query(query_locaux, conn)
print("1. Repartition du parc immobilier")
display(df_locaux)

plt.figure(figsize=(10, 5))
sns.set_theme(style="whitegrid")
sns.barplot(data=df_locaux, x='Type', y='Total', hue='Disponibilite', palette='Set2')
plt.title('Repartition des locaux par type et disponibilite')
plt.ylabel('Nombre de locaux')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

# Requete 2 : Analyse multi-tables (Contrats, Paiements et Taux de recouvrement)
query_recouvrement = """
SELECT 
    l.gestionnaire AS Gestionnaire,
    c.type_contrat AS Type_Contrat,
    COUNT(DISTINCT c.id) AS Nb_Contrats,
    ROUND(SUM(e.montant_du), 0) AS Total_Exigible_FCFA,
    ROUND(SUM(CASE WHEN e.statut = 'PAYEE' THEN e.montant_du ELSE 0 END), 0) AS Total_Recouvre_FCFA,
    ROUND((SUM(CASE WHEN e.statut = 'PAYEE' THEN e.montant_du ELSE 0 END) * 100.0) / SUM(e.montant_du), 1) AS Taux_Recouvrement_Pct
FROM contrats_contrat c
JOIN patrimoine_local l ON c.local_id = l.id
JOIN paiements_echeance e ON e.contrat_id = c.id
GROUP BY l.gestionnaire, c.type_contrat;
"""
df_recouvrement = pd.read_sql_query(query_recouvrement, conn)
print("\n2. Taux de recouvrement par type de contrat et gestionnaire")
display(df_recouvrement)

### 4. Cartographie Geospatiale Interactive des Locaux (Folium)

In [ ]:
import folium

query_gps = """
SELECT reference, type_local, localisation, surface_m2, etat_physique, est_libre, latitude, longitude 
FROM patrimoine_local 
WHERE latitude IS NOT NULL AND longitude IS NOT NULL;
"""
df_gps = pd.read_sql_query(query_gps, conn)

# Centre de la carte : Campus CROUS Thies
lat_centre = df_gps['latitude'].mean() if not df_gps.empty else 14.7892
lon_centre = df_gps['longitude'].mean() if not df_gps.empty else -16.9261

carte = folium.Map(location=[lat_centre, lon_centre], zoom_start=16, tiles='OpenStreetMap')

for _, row in df_gps.iterrows():
    couleur = 'green' if row['est_libre'] == 1 else ('orange' if row['etat_physique'] in ('EN_TRAVAUX', 'DEGRADE') else 'red')
    popup_html = f"""
    <b>Local :</b> {row['reference']}<br>
    <b>Type :</b> {row['type_local']}<br>
    <b>Surface :</b> {row['surface_m2']} m²<br>
    <b>Etat :</b> {row['etat_physique']}<br>
    <b>Statut :</b> {'Disponible' if row['est_libre'] == 1 else 'Occupe'}
    """
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{row['reference']} ({row['type_local']})",
        icon=folium.Icon(color=couleur, icon='home')
    ).add_to(carte)

print(f"Affichage de la carte interactive : {len(df_gps)} locaux cartographies sur le campus")
display(carte)

### 5. Modele Predictif BDA (Machine Learning) : Risque de Retard de Paiement

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Extraction des echeances avec variables explicatives
query_ml = """
SELECT 
    e.montant_du,
    l.surface_m2,
    c.duree_mois,
    CASE WHEN l.gestionnaire = 'CROUS_T' THEN 1 ELSE 0 END AS est_gestion_crous,
    CASE WHEN c.type_contrat = 'BAIL_COMMERCIAL' THEN 1 ELSE 0 END AS est_bail_commercial,
    CASE WHEN e.statut IN ('EN_RETARD', 'EXIGIBLE') THEN 1 ELSE 0 END AS retard_label
FROM paiements_echeance e
JOIN contrats_contrat c ON e.contrat_id = c.id
JOIN patrimoine_local l ON c.local_id = l.id;
"""
df_ml = pd.read_sql_query(query_ml, conn)

X = df_ml[['montant_du', 'surface_m2', 'duree_mois', 'est_gestion_crous', 'est_bail_commercial']]
y = df_ml['retard_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
score = model.score(X_test, y_test)

print(f"Precision du modele de prediction des retards : {score * 100:.1f} %")
y_pred = model.predict(X_test)
print("\nRapport de classification :\n", classification_report(y_test, y_pred))

# Importance des variables explicatives
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(x=importances.values, y=importances.index, palette='Blues_r')
plt.title('Importance des facteurs influencant le risque de retard de paiement')
plt.xlabel('Importance relative')
plt.tight_layout()
plt.show()

### 6. Export Automatique des Rapports BDA (Excel & CSV)

In [ ]:
df_locaux.to_excel('rapport_patrimoine_crous.xlsx', index=False)
df_recouvrement.to_csv('rapport_recouvrement_comptable.csv', index=False)

print("Rapports generes avec succes dans l'environnement Colab :")
print("- rapport_patrimoine_crous.xlsx")
print("- rapport_recouvrement_comptable.csv")

### 7. Deploiement Cloud avec Tunnel Cloudflare (Liens Directs)

In [ ]:
import subprocess
import time
import re

print("1. Lancement du serveur Django en arriere-plan...")
backend_proc = subprocess.Popen(["python", "vcn_backend/manage.py", "runserver", "0.0.0.0:8000"])
time.sleep(2)

print("2. Configuration du tunnel Cloudflare...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

url_trouvee = None
for line in tunnel_proc.stderr:
    match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', line)
    if match:
        url_trouvee = match.group(1)
        break

if url_trouvee:
    print("\n" + "="*60)
    print("LIENS DIRECTS D'ACCES A L'APPLICATION SYLOC-T :\n")
    print(f"- Documentation Swagger interactive : {url_trouvee}/api/docs/")
    print(f"- Administration Django (admin/admin): {url_trouvee}/admin/")
    print(f"- Statistiques BDA (JSON)            : {url_trouvee}/api/public/stats/")
    print(f"- Vitrine officielle CROUS-T         : {url_trouvee}/api/public/vitrine/")
    print(f"- Liste des locaux disponibles       : {url_trouvee}/api/public/locaux/")
    print("="*60)
    print("\nLe serveur est actif dans le Cloud. Laissez cette cellule tourner pendant vos tests.")
    tunnel_proc.wait()
else:
    print("Impossible de recuperer l'URL du tunnel Cloudflare.")